# MetworkPy *Mycobacterium Tuberculosis* Transcription Factors Example

An example of using MetworkPy to analyze the targets of transcription factors (TFs) in *Mycobacterium tuberculosis* (Mtb). Mtb is an adaptable pathogen, able to survive under both significant host pressure and antibiotic treatment (with the shortest course taking at least 4 months of multiple antibiotics). In order to be able to adapt to shifting host pressures, and environmental changes, Mtb must be able to sense these changes, and respond to them. One important response mechanism is via TFs which alter the expression level of target genes in order to improve the pathogen's ability to survive. 

In order to investigate one of these TFs (specifically ArgR, a known regulator of arginine biosynthesis) we will be using data from Rustad et al. 2014 (full citation below) which explored the TF regulatory network in Mtb. Specially, they generated a library of TF overexpression strains (referred to here as TFOE) and mapped the genes which had altered expression upon the induction of the TFs using microarrays. This allowed them to identify a large number of targets of the TFs, and also generated expression data for the TFOE strains. We will use both of these to analyze the impact of the TFs on Mtb metabolism.

Data about the Mtb TFs is from: Rustad, T.R., Minch, K.J., Ma, S. et al. Mapping and manipulating the Mycobacterium tuberculosis transcriptome using a transcription factor overexpression-derived regulatory network. Genome Biol 15, 502 (2014). [doi: 10.1186/s13059-014-0502-3](https://doi.org/10.1186/s13059-014-0502-3). 
  
Genome scale metabolic model (GSMM) from: A systematic evaluation of Mycobacterium tuberculosis Genome-Scale Metabolic Networks
López-Agudelo VA, Mendum TA, Laing E, Wu H, Baena A, et al. (2020) A systematic evaluation of Mycobacterium tuberculosis Genome-Scale Metabolic Networks. PLOS Computational Biology 16(6): e1007533. [https://doi.org/10.1371/journal.pcbi.1007533](https://doi.org/10.1371/journal.pcbi.1007533)

## Get Data

This example uses data from "Mapping and manipulating the Mycobacterium tuberculosis transcriptome using a transcription factor overexpression-derived regulatory network" by Rustad et al. which needs to be downloaded before this example can be run. 

The data can be obtained from [https://pmc.ncbi.nlm.nih.gov/articles/PMC4249609/](https://pmc.ncbi.nlm.nih.gov/articles/PMC4249609/), from the `Additional file 1: Table S1.` link right below the Acknowledgements section. Please download the file, and update the `tfoe_data_path` in the cell below this one to point to where you downloaded the file.

In [1]:
# Relative path to the TFOE data, this can be retrieved from
# https://pmc.ncbi.nlm.nih.gov/articles/PMC4249609/
tfoe_data_path = "13059_2014_502_MOESM1_ESM.xlsx"

## Setup

In [2]:
# Import needed Packages
# Standard library imports
from collections import (
    defaultdict,
)  # Used for counting metabolite-reaction associations
import warnings  # Used for suppressing some irrelevant warnings

# Import main package
import metworkpy

# For handling data
import pandas as pd

# For dealing with Metabolic Models (FBA, pFBA, FVA, Flux Sampling, etc.)
import cobra

# For network evaluation/exploration
import networkx as nx

### Data Extraction

We will extract the log2(fold-change) and false-discovery corrected p-values from the supplementary data associated with Rustad et al. 2014.

In [3]:
%%time
# Read in the log2(fold-change) and FDR corrected p-value data from the excel file
tfoe_fc = pd.read_excel(
    tfoe_data_path,
    sheet_name="SupplementaryTableS2",
    skiprows=list(range(8)) + [9],
    usecols="A,E:HB",
    index_col=0,
)
tfoe_pval = pd.read_excel(
    tfoe_data_path,
    sheet_name="SupplementaryTableS2",
    skiprows=list(range(8)) + [9],
    usecols="A,HC:OZ",
    index_col=0,
)
# Since the p-value column headers are read in after the foldchange column headers, they have a .1 appended to them
# even though the fold-change column headers aren't in the tfoe_pval final dataframe, so we remove this suffix
tfoe_pval.columns = tfoe_pval.columns.str.removesuffix(".1")

CPU times: user 36 s, sys: 311 ms, total: 36.3 s
Wall time: 36.4 s


Using the log2(fold-change) and the p-values we can define a set of gene regulatory targets for each TF, in this case we will be focusing on ArgR (Rv1657), so we will find its targets. Here we define targets as those genes showing a greater than 2-fold change (in either direction) with an adjusted p-value of less than 0.05. 

In [4]:
argr_targets = list(
    tfoe_fc[
        (tfoe_fc["Rv1657"].abs() >= 1.0) & (tfoe_pval["Rv1657"] <= 0.05)
    ].index
)

### Metabolic Model

Read in the GSMM from López-Agudelo et al. 2020, which has had the values of its exchange reactions modified to more closely match the media conditions expected of the TFOE strains.

In [5]:
# Set the default solver to be the hybrid solver which doesn't require a license
# but can still handle MILP problems. If this doesn't work, highspy and osqp need to
# be installed, so run
# `pip install highspy osqp`
# in the virtual environment
cobra.Configuration().solver = "hybrid"

# Make use of helper function from MetworkPy which just wraps COBRApy's io module into a single function
gsmm = metworkpy.read_model("iEK1011_v2_7H9_ADC_glycerol.json")

## Network Creation

MetworkPy enables the creation of a few different kinds of networks, including
- Stoichiometric Connectivity Networks: Connect reactions/metabolites which
  are connected by their stoichiometric equations. In the case of the full metabolic
  stoichiometric connectivity network this means connecting reactions to their metabolic substrates.
  This creates a bipartite network (meaning a network with two sets of nodes, where all edges are between
  the node sets), with a set of reaction nodes, and a set of metabolite nodes. This network
  can be projected onto the reactions or metabolites creating the reaction stoichiometric connectivity networks
  and metabolite stoichiometric connectivity networks, respectively.
- Mutual information networks: Networks where edge weights represent the mutual information between
  the nodes. Mutual information is a measure of statistical dependence, simmilar to correlation
  but able to capture more types of dependence than just linear (or monotonic) relationships. This
  can be for the fluxes of different reactions in the GSMM, but can also be applied much more generally.

### Metabolic Network

For the metabolic networks it can be useful to remove reactions/metabolites which are irrelevant or which connect a large portion of the metabolism (such as pseudoreactions, and metabolites which are used for many otherwise unrelated reactions such as H20 and ATP). We will create a list of reactions and metabolites we want to exclude from the network. 

In [6]:
%%time
# Create a list of reactions to excude
reactions_to_exclude = []
# These subsystems contain pseudoreactions, or other reactions we don't want to include in the network
subsystems_to_exclude = [
    "Biomass and maintenance functions",
    "Intracellular demand",
    "Extracellular exchange",
]
# Create a dict to record how many reactions metabolites would be connected to
# so as to remove the most highly connected metabolites
metabolite_rxn_count_dict: dict[str, int] = defaultdict(int)
model_reactions = set()
for rxn in gsmm.reactions:
    if rxn.subsystem in subsystems_to_exclude:
        reactions_to_exclude.append(rxn.id)
        continue
    model_reactions.add(rxn.id)
    for metabolite in rxn.metabolites:
        metabolite_rxn_count_dict[metabolite.id] += 1


# Also exclude highly connected metabolites
metabolites_to_exclude = list(
    set(
        map(
            lambda t: t[0],  # Need to extract the id from the tuples produced
            sorted(  # Sort the metabolites based on how many reactions they are connected to
                metabolite_rxn_count_dict.items(),  # Using item tuples to allow for sorting, and then id extraction
                key=lambda i: i[1],  # Key the sorting by the reaction count
                reverse=True,  # Want the most highly connected at the front of the list
            )[:10],  # Excluding the top 10 most used metabolites
        )
    )
)

# Create a list of nodes to exclude
nodes_to_exclude = reactions_to_exclude + metabolites_to_exclude

CPU times: user 1.97 ms, sys: 57 μs, total: 2.03 ms
Wall time: 2.04 ms


In [7]:
metabolic_network = metworkpy.create_metabolic_network(
    model=gsmm,
    weighted=False,
    directed=False,
    nodes_to_remove=nodes_to_exclude,
)

### Reaction Network

In [8]:
# The biparite network can be projected onto the reactions only
reaction_network = metworkpy.bipartite_project(
    network=metabolic_network,
    # Project onto the reaction nodes, but filter for reactions which actually ended up in the network
    node_set=[
        r for r in gsmm.reactions.list_attr("id") if r not in nodes_to_exclude
    ],
    # Using undirected networks for simplicity
    directed=False,
)

# Or can be created directly using a helper function
reaction_network = metworkpy.create_reaction_network(
    model=gsmm,
    nodes_to_remove=nodes_to_exclude,
    # Using undirected and unweighted networks for simplicity
    directed=False,
    weighted=False,
)

### Metabolite Network

The network can similarly be projected onto the metabolite nodes

In [9]:
metabolite_network = metworkpy.create_metabolite_network(
    model=gsmm,
    nodes_to_remove=nodes_to_exclude,
    # Using undirected and unweighted networks for simplicity
    directed=False,
    weighted=False,
)

### Mutual Information Network

A mutual information network for the fluxes of the reactions in the GSMM can also be created. This makes of flux sampling to find the flux distributions of reactions in the model, and then calculates the mutual information between each pair of reactions in the model. This results in a network with nodes for every reaction in the network, and with edges which are weighted by the mutual information between the fluxes of the reactions that the edge is connecting.

In [10]:
%%time
mi_network = metworkpy.create_mutual_information_network(
    model=gsmm,
    n_samples=100,
    progress_bar=False,
    processes=-1,
    clip=True,
)

CPU times: user 5min 8s, sys: 7.29 s, total: 5min 16s
Wall time: 4min 49s


## Network Analysis

### Centrality

Since the networks created above are just NetworkX Graph objects, we can analyze them using NetworkX. For example, we can find the closeness centrality of the nodes in the reaction and metabolite networks.

In [11]:
reaction_closeness = pd.Series(nx.closeness_centrality(reaction_network))
metabolite_closeness = pd.Series(nx.closeness_centrality(metabolite_network))

Which allows us to see the most central nodes in both networks:

In [12]:
print("Top Closeness of Reactions in the iEK1011_v2 Mtb GSMM:")
reaction_closeness.sort_values(ascending=False).head(10)

Top Closeness of Reactions in the iEK1011_v2 Mtb GSMM:


GLUSy      0.429085
GSEALD     0.426336
ASNS1      0.422445
MCBTS3     0.420609
FAS181     0.418460
FAO2       0.417148
GMPS2      0.416658
DESAT16    0.415520
ACS        0.415357
NADS2      0.413582
dtype: float64

In [13]:
print("Top Closeness of Metabolites in the iEK1011_v2 Mtb GSMM:")
metabolite_closeness.sort_values(ascending=False).head(10)

Top Closeness of Metabolites in the iEK1011_v2 Mtb GSMM:


glu__L[c]    0.409393
nadph[c]     0.404035
nadp[c]      0.404035
amp[c]       0.400600
pyr[c]       0.377622
o2[c]        0.374777
accoa[c]     0.374777
gln__L[c]    0.369670
hdca[c]      0.367395
nh4[c]       0.366043
dtype: float64

For the mutual information network, the edge weights correspond to association strengths rather than distances, so closeness (which is based on shortest path distances) doesn't capture the centrality, but other metrics such as PageRank (or Katz Centrality, Eigenvector Centrality, Information Centrality, Current flow betweenness/closeness, etc.) can work with association strengths instead of distances.

In [14]:
reaction_pagerank = pd.Series(
    nx.pagerank(mi_network, weight="weight", tol=1e-6, max_iter=500)
)

In [15]:
print(
    "Top PageRank Reactions in the iEK1011_v2 Mtb Flux Mutual Information Network:"
)
reaction_pagerank.sort_values(ascending=False).head(10)

Top PageRank Reactions in the iEK1011_v2 Mtb Flux Mutual Information Network:


CBIAT        0.018121
ADOCBLabc    0.009669
NAPRT        0.006576
BARB         0.005680
DMBI         0.004377
DBTS         0.003699
NNDMBRT      0.003218
CYRDAR2      0.003218
COB2         0.003052
ADOCBLS      0.003036
dtype: float64

### Density/Enrichment

MetworkPy also includes some additional methods aimed at facilitating the analysis of nodes which are targeted. For example, we can see if there are concentrations of target genes around the reactions in the network. MetworkPy can either find the density or the enrichment of targets in a neighborhood around the nodes in the network. The density is the number of targets in a neighborhood, normalized by the size of the neighborhood, and the enrichment is calculated using a hypergeometric test for enrichment.

The reactions with the highest density of ArgR targets in their neighborhoods are:

In [16]:
# Find the target density
reaction_argr_target_density = metworkpy.network.gene_target_density(
    metabolic_network=reaction_network,
    # Pass the metabolic model which is used for finding the genes associated
    # with each reaction based on the gene-protein-reaction rules of the model
    metabolic_model=gsmm,
    # The targets to find the density for
    gene_targets=argr_targets,
    # Defines the maximum distance from a node to still be considered
    # in the nodes neighborhood
    radius=1,
)
print("Most target dense reaction neighborhoods for ArgR:")
argr_most_dense_reactions = reaction_argr_target_density.sort_values(
    ascending=False
).head(10)
argr_most_dense_reactions

Most target dense reaction neighborhoods for ArgR:


ACGK      0.750000
ORNt      0.333333
ORNDC     0.285714
OCBT      0.272727
ARGDC     0.250000
ARGt5r    0.250000
ASPCT     0.125000
ARGSL     0.100000
ASPK      0.076923
ORNTAC    0.074074
dtype: float64

In [17]:
# Find the subsystems which the above reactions are in
for rxn in argr_most_dense_reactions.index:
    rxn_obj = gsmm.reactions.get_by_id(rxn)
    print(
        f"Reaction: {rxn}, name: {rxn_obj.name}, Subsystem: {rxn_obj.subsystem}"
    )

Reaction: ACGK, name: Acetylglutamate kinase, Subsystem: Arginine and Proline Metabolism
Reaction: ORNt, name: Ornithine transport via diffusion, Subsystem: Transport
Reaction: ORNDC, name: Ornithine Decarboxylase, Subsystem: Spermidine biosynthesis
Reaction: OCBT, name: Ornithine carbamoyltransferase, Subsystem: Arginine and Proline Metabolism
Reaction: ARGDC, name: Arginine decarboxylase, Subsystem: Alanine, Aspartate, and Glutamate Metabolism
Reaction: ARGt5r, name: ARGtiDF, Subsystem: Transport
Reaction: ASPCT, name: Aspartate carbamoyltransferase, Subsystem: Purine and Pyrimidine Biosynthesis
Reaction: ARGSL, name: Argininosuccinate lyase, Subsystem: Alanine, Aspartate, and Glutamate Metabolism
Reaction: ASPK, name: Aspartate kinase, Subsystem: Glycolysis/Gluconeogenesis
Reaction: ORNTAC, name: Ornithine transacetylase, Subsystem: Arginine and Proline Metabolism


In [18]:
reaction_argr_enrichment = metworkpy.network.gene_target_enrichment(
    metabolic_network=reaction_network,
    metabolic_model=gsmm,
    gene_targets=argr_targets,
    # Specifically looking for enrichment, not depletion
    alternative="greater",
    # Method can also return the odds-ratio, but we'll use the p-values
    metric="p-value",
    # Same radius as in the density above
    radius=1,
)
print("Most enriched reaction neighborhoods for ArgR:")
argr_most_enriched_reactions = reaction_argr_enrichment.sort_values(
    ascending=True
).head(10)
argr_most_enriched_reactions

Most enriched reaction neighborhoods for ArgR:


ACGK      8.152869e-07
OCBT      3.293385e-05
ORNTAC    2.274025e-04
ORNt      6.100578e-04
ORNDC     8.512529e-04
ARGDr     3.086108e-03
ASPCT     4.721018e-03
ACOTA     5.078460e-03
ORNTA     5.884653e-03
CBPS      7.076765e-03
dtype: float64

In [19]:
# Find the subsystems which the above reactions are in
for rxn in argr_most_enriched_reactions.index:
    rxn_obj = gsmm.reactions.get_by_id(rxn)
    print(
        f"Reaction: {rxn}, name: {rxn_obj.name}, Subsystem: {rxn_obj.subsystem}"
    )

Reaction: ACGK, name: Acetylglutamate kinase, Subsystem: Arginine and Proline Metabolism
Reaction: OCBT, name: Ornithine carbamoyltransferase, Subsystem: Arginine and Proline Metabolism
Reaction: ORNTAC, name: Ornithine transacetylase, Subsystem: Arginine and Proline Metabolism
Reaction: ORNt, name: Ornithine transport via diffusion, Subsystem: Transport
Reaction: ORNDC, name: Ornithine Decarboxylase, Subsystem: Spermidine biosynthesis
Reaction: ARGDr, name: Arginine deiminase, Subsystem: Arginine and Proline Metabolism
Reaction: ASPCT, name: Aspartate carbamoyltransferase, Subsystem: Purine and Pyrimidine Biosynthesis
Reaction: ACOTA, name: Acetylornithine transaminase, Subsystem: Arginine and Proline Metabolism
Reaction: ORNTA, name: Ornithine transaminase, Subsystem: Arginine and Proline Metabolism
Reaction: CBPS, name: Carbamoyl-phosphate synthase (glutamine-hydrolysing), Subsystem: Purine and Pyrimidine Biosynthesis


## Subnetworks

### Metabolite Synthesis Networks

MetworkPy also includes facilities for finding the networks of reactions and/or genes which are required for producing a specific metabolite. Finding this for all of the metabolites in the Model can take a long time, so we will filter this down to only arginine. The metabolite synthesis networks are returned in the form of a dataframe with metabolites represented by columns, and genes (or reactions) by rows. A value of True at row i column j in the dataframe represents that the gene i is required for the synthesis of metabolite j. Below we use `find_metabolite_synthesis_network_genes`, but there is also `find_metabolite_synthesis_network_reactions` which will find the reactions required to produce a metabolite instead of the genes.

In [20]:
%%time
metabolite_synthesis_networks = (
    metworkpy.find_metabolite_synthesis_network_genes(
        model=gsmm,
        metabolites=["arg__L[c]"],
        method="essential",
        essential_proportion=0.1,
        progress_bar=False,
    )
)
arg_synthesis_genes = list(
    metabolite_synthesis_networks[
        metabolite_synthesis_networks["arg__L[c]"]
    ].index
)

CPU times: user 140 ms, sys: 174 ms, total: 313 ms
Wall time: 10.4 s


These synthesis networks could be passed into other bioinformatic pipelines such as gene set variability analysis in order to create. 

### Metabolite Consuming Networks

In addition to finding the genes and/or reactions required to generate a metabolite, MetworkPy can also find the reactions which have altered flux upon depletion of a metabolite, called consuming networks by MetworkPy.

In [21]:
%%time
metabolite_consuming_networks = (
    metworkpy.metabolites.find_metabolite_consuming_network_reactions(
        model=gsmm,
        metabolites=["arg__L[c]"],
        reaction_proportion=0.1,
        progress_bar=False,
    )
)
arg_consuming_reactions = list(
    metabolite_consuming_networks[
        metabolite_consuming_networks["arg__L[c]"]
    ].index
)

CPU times: user 705 ms, sys: 691 ms, total: 1.4 s
Wall time: 46.5 s


## Divergence

In order to investigate metabolic perturbations, MetworkPy includes facilities for evaluating flux differences using a statistical distance measure called divergence. This approach uses flux sampling (and so doesn't require a metabolic objective function, though constraints on a metabolic objective function can be applied), and quantifies the distance between the flux distributions of a base model, and a perturbed model. Divergence can measure the distance between multidimensional distributions as well as between 1D distributions allowing it to be applied to individual reactions, subsystems, pathways, and metabolite subnetworks. 

Specifically, MetworkPy can estimate the Kullback-Leibler and the Jensen-Shannon divergence between samples from two distributions. There are a variety of perturbations that can be applied to GSMMs, and as long as the resulting models can be flux sampled the divergence can be calculated. We will examine two perturbations, specifically gene knockout and context-specific model constraints.

### KO Divergence

MetworkPy includes a helper function for evaluating the divergences caused by single gene knockouts in a GSMM. It takes in a GSMM, an optional list of genes to knockout, and divergence groups. The divergence groups describe which sets of reactions should be evaluated in terms of divergence. This takes the form of a dictionary from a name (used in the output dataframe) to a list of reaction ids.

In [22]:
%%time
# Create divergence groups from the subsystems in the model
divergence_targets = defaultdict(set)
for rxn in gsmm.reactions:
    divergence_targets[rxn.subsystem].add(rxn.id)
# Convert the sets of reaction ids to lists
divergence_targets = {k: list(v) for k, v in divergence_targets.items()}

ko_divergence = metworkpy.divergence.ko_divergence(
    model=gsmm,
    target_networks=divergence_targets,
    # By default this method knocks out each gene in the model, but we can subset it
    # to speed this up
    genes_to_ko=["Rv3502c", "Rv0417", "Rv1658", "Rv2181"],
    divergence_type="kl",  # Can select between kl (Kullback-Leibler) and js (Jensen-Shannon)
    sample_count=100,
    progress_bar=False,
    sampler_seed=10928312,
    processes=-1,
    # Having samples that are all identically 0,
    # which can happen when a gene is knocked out,
    # caused problems with the divergence estimation,
    # so we can add some "jitter", that is some
    # small random noise to avoid this
    jitter=1e-7,
    jitter_seed=123901720974,
)

CPU times: user 18min 44s, sys: 2.81 s, total: 18min 47s
Wall time: 13min 38s


### Condition Specific Model

Another perturbation that can be evaluated is that of condition specific metabolic models. GSMM are intentionally very general, not capturing a specific condition but instead trying to combine all of the metabolic activity that could be happending across a range of conditions. In order to tailor these models to a specific type of condition, for example a specific cell type in the case of multicellular organisms, a variety of methods have been developed to generate models that are speicific to a particular condition. In general these methods use some -omic data to constrain the model so that it aligns with the condition of interest. 

The differences between a condition specific metabolic model and the base model (or between two condition speicif models) can them be evaluated to investigate the metabolic perculiarities created by a specific condition. As long as the method of generating a condition specific metabolic model creates a COBRApy model which can be flux sampled, the divergence approach can be applied to investigate these metabolic changes. 

MetworkPy includes an implementation of the iMAT algorithm of Shlomi et al. 2008/Zur et al. 2010 to generate these condition specific metabolic models. This method uses gene expression data to constrain the GSMM. Specifically, it starts from gene expression weights, which have been trinarized into -1 for low expression genes, 1 for high expression genes, and 0 for the remaining genes. These gene weights are then converted into reaction weights using the gene-protein-reaction rules of the GSMM. iMAT also defines active and inactive reactions based on flux, with active reactions having a flux above a threshold (called epsilon), and inactive reactions have no flux (or flux below a cutoff). The reaction weights, and reaction activities are then combined into the iMAT objective function, which tries to find the best possible match, with reactions with high weights being active, and reactions with low weights being inactive. 

**References**
- Shlomi, T., Cabili, M. N., Herrgård, M. J., Palsson, B. Ø., & Ruppin, E. (2008). Network-based prediction of human tissue-specific metabolism. Nature Biotechnology, 26(9), 1003–1010. https://doi.org/10.1038/nbt.1487
- Zur, H., Ruppin, E., & Shlomi, T. (2010). iMAT: An integrative metabolic analysis tool. Bioinformatics (Oxford, England), 26(24), 3140–3142. https://doi.org/10.1093/bioinformatics/btq602

#### Gene Weights

The first step in the iMAT process is to find the gene weights. Here we will be using differential expression, but you could alternatively use expression quantiles (i.e. the bottom 15% of genes are given weight -1, and top 15% weight of 1). 

In [23]:
# Set gene weights based on differential expression, with genes with >1 log2(fold-change) expression
# relative to median being given a weight of +1, and genes with a <-1 log2(fold-change) expression
# relative to median being given a weight of -1
argr_gene_expression_foldchange = tfoe_fc["Rv1657"]
# Default weight is 0.0
gene_weights = pd.Series(0.0, index=argr_gene_expression_foldchange.index)
# Set the high expression weights
gene_weights[argr_gene_expression_foldchange >= 1.0] = 1.0
# Set the low expression weights
gene_weights[argr_gene_expression_foldchange <= 1.0] = -1.0

#### Reaction Weights

Next, these gene weights need to be converted into reaction weights. This is done through the gene-protein-reaction rules of the GSMM. These are Boolean rules describing which genes are required for a reaction to be active. These rules include the Boolean AND and OR operators, which for the trinarized gene weights we define to be minimum and maximum respectively. 

In [24]:
with warnings.catch_warnings():
    # MetworkPy will warn if there are genes in the Model not found in
    # gene weights, this can be useful for catching issues in general,
    # but here we don't care so we filter them out
    warnings.simplefilter("ignore")
    reaction_weights = metworkpy.gene_to_rxn_weights(
        model=gsmm, gene_weights=gene_weights
    )

#### Condition Specific Metabolic Model

We can now use these reaction weights to generate a condition specific metabolic model. There are several methods in MetworkPy for generating a condition-specific metabolic model from the iMAT optimization problem, here we will be using "fva", but you can see the API documentation for the other methods. 

In [25]:
%%time
with warnings.catch_warnings():
    warnings.simplefilter(
        "ignore"
    )  # The iMAT method can occasionally run into issues with infeasiblity, but it will still work
    argr_imat = metworkpy.imat.generate_model(
        model=gsmm,  # Base model to use
        rxn_weights=reaction_weights,  # Weights of the reactions,
        method="fva",
        epsilon=1.0,  # Threshold for considering reactions active
        threshold=0.01,  # Threshold for considering reactions inactive
        objective_tolerance=0.95,  # Tolerance for FVA method
    )

CPU times: user 17.7 s, sys: 518 ms, total: 18.2 s
Wall time: 4min 25s


#### Divergence

Now that we have an iMAT model we can calculate the divergence between the iMAT model and the base GSMM.

In [26]:
%%time
# Generate flux samples from both models using cobra
base_flux_samples = cobra.sampling.sample(gsmm, 100, processes=-1)
argr_flux_samples = cobra.sampling.sample(argr_imat, 100, processes=-1)

CPU times: user 8min 37s, sys: 1.52 s, total: 8min 38s
Wall time: 4min 32s


In [27]:
# Create a list of the sets of reactions we want to calculate the divergence for
# We will just be using the individual reactions of the model
divergence_targets = {}
for rxn in gsmm.reactions:
    divergence_targets[rxn.id] = [rxn.id]

# Calculate the divergence between every reaction in the ArgR and Base Models
divergence_df = metworkpy.divergence.calculate_divergence_grouped(
    base_flux_samples,
    argr_flux_samples,
    divergence_groups=divergence_targets,
    divergence_type="kl",
    calculate_pvalue=False,
    jitter=1e-7,
    jitter_seed=102947120497,
    processes=-1,
)

As mentioned earlier, divergence can also be applied to larger sets of reactions, such as subsystems and pathways. It should be noted though that divergence values are generally larger for higher dimensional distributions, i.e. the larger the subsystem/pathway the higher the divergence. To account for this divergence of subsystems/pathways could be normalized, in the case of the TFOE strains we could generate iMAT models for all of them, and then normalize the divergence across all of these models. Another approach is to make use of permutation testing, to evaluate the significance of the divergence rather than its magnitude directly. To do this you can add the argument `calculate_pvalue=True` to most of the divergence methods and have two dataframes returned instead of 1, the first being the divergence values, the second being the p-values calculated using permutation testing. Note that permutation testing can take a significant amount of time, as the divergence calculation is relatively computationally expensive.  